###  What is ReAct?

**ReAct = Reason + Act** - an AI agent that first **thinks**, then **acts**, then **observes the result** - and repeats this until it finds the answer.



###  The Loop - Step by Step

**1. It receives a question from the user**
For example: *"Are warfarin + aspirin for a woman who is 32?"*

**2. It sends the question to the LLM (Mistral)**
The model receives:
- a system prompt (instructions on how to behave)
- the conversation history
- a list of available tools

**3. The model THINKS**
The model asks itself: *"What do I already know? What am I still missing?"*

**4. The model ACTS - it calls a tool**
Instead of answering right away, the model calls one of 5 tools:
- `lookup_interactions` - finds interactions between drugs
- `lookup_population_warnings` - finds warnings for specific groups (pregnancy, children, elderly)
- `search_drug_kb` - searches the FDA knowledge base
- `flag_severity` - classifies how serious each finding is
- `summarize_evidence` - builds the final report

**5. The tool returns a result (Observe)**
The result is added to the conversation history - the model can now "see" what it found.

**6. Go back to Step 2 - up to 15 times**
The agent repeats the Think → Act → Observe loop, collecting more evidence each time.

**7. Done - the model answers without calling any tool**
When the model decides it has enough data - instead of calling a tool, it writes the **final answer** (a report with the full drug interaction analysis).

**8. If it hits the limit of 15 iterations**
It returns whatever it collected with the message: *"Max iterations reached"* - meaning it ran out of steps before finishing.



###  What does it return at the end?

```python
{
  "trace":          [...],  # full log: what it thought, did, and observed
  "final_response": "...",  # the final text answer
  "tool_results":   [...]   # raw results from all tool calls
}
```



**In short:** the agent works like a detective - it never guesses. It collects evidence step by step from the FDA database, and only writes the report when it has the full picture.

![The ReAct agent ](../data/react_diagram.png)

### Why does this work better than "one big prompt"?

- A **plain LLM** hallucinates drug interactions. ReAct forces every claim to cite a `source_id` from a real FDA label.
- **Single-shot RAG** isn't enough - a query like *KETOCONAZOLE × aspirin x child* needs three distinct searches. The agent decides how many times and what to search for.
- **The ReAct** agent dynamically picks which tool to call next based on what it has already learned.

**Key insight:** `max_iterations=15` is a **safety net, not a target**. Typically the loop ends after 4–8 iterations, once the model concludes it has gathered sufficient evidence.

### Cell 1 - Initialize All Components

This cell connects all three components the ReAct agent needs to operate.

**1. PostgreSQL connection**
Reads the database credentials directly from OpenTofu outputs - no hardcoded values.
Connects securely with `sslmode="require"`.

**2. Embeddings client**
Calls the **BGE Multilingual Gemma2** model on Scaleway Generative APIs (serverless, pay-per-token).
This client converts text queries into 768-dimensional vectors for similarity search.

**3. LLM client**
Connects to **Mistral Small 3.2** via Scaleway Generative APIs.
This is the "brain" of the agent - it reasons, decides which tool to call, and writes the final report.

**4. ToolKit**
All three components are passed into `ToolKit`, which wires them together into the 5 tools the agent will use.

In [ ]:
import os
import sys
import subprocess
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
sys.path.insert(0, "..")

# Database connection
import psycopg

pg_dir = "../iac_snippets/postgres"


def get_pg_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=pg_dir, capture_output=True, text=True)
    return r.stdout.strip()


conn = psycopg.connect(
    host=get_pg_output("host"),
    port=int(get_pg_output("port")),
    dbname=get_pg_output("database"),
    user=get_pg_output("user"),
    password=get_pg_output("password"),
    sslmode="require",
)

# Embeddings client (Scaleway Generative APIs, serverless)
from src.embeddings import EmbeddingsClient

embedding_client = EmbeddingsClient(
    client=OpenAI(base_url="https://api.scaleway.ai/v1", api_key=os.environ["SCW_SECRET_KEY"]),
    model="bge-multilingual-gemma2",
    dimensions=768,
)

# LLM client
llm_client = OpenAI(
    base_url="https://api.scaleway.ai/v1",
    api_key=os.environ["SCW_SECRET_KEY"],
)

# Create toolkit
from src.tools import ToolKit

toolkit = ToolKit(conn=conn, embeddings_client=embedding_client, llm_client=llm_client)

print("All components ready.")

### Baseline - Plain Mistral, No Tools

Before running the ReAct agent, we ask the same clinical question directly to Mistral - with no tools, no database, no FDA labels.

> *"A patient is taking KETOCONAZOLE and ibuprofen. The patient is 15 years old."*

The model answers from memory alone - no sources, no citations, no verification.

In [ ]:
CANONICAL_QUERY = (
    "A patient is taking KETOCONAZOLE and ibuprofen. "
    "The patient is 15 years old. "
    "What are the drug interactions and population-specific warnings? "
    "List all findings with severity and cite your sources."
)

# Baseline - plain Mistral, no tools
baseline = llm_client.chat.completions.create(
    model="mistral-small-3.2-24b-instruct-2506",
    messages=[{"role": "user", "content": CANONICAL_QUERY}],
    temperature=0.05,
    max_tokens=1000,
)

print("=== BASELINE (plain Mistral, no tools) ===")
print(baseline.choices[0].message.content)

## ReAct Agent: Full Think -> Act -> Observe Trace

Now the same query through the ReAct loop with all five tools.

In [ ]:
from src.react_loop import run_react_loop

result = run_react_loop(
    query=CANONICAL_QUERY,
    toolkit=toolkit,
    llm_client=llm_client,
)

print(f"Agent completed in {len(result['trace'])} iterations.")

In [ ]:
# Render the full trace
print("=== REACT AGENT TRACE ===")
print()
for i, entry in enumerate(result["trace"]):
    print(f"=== Iteration {i + 1} ===")
    print(f"THINK: {entry['think'][:200]}")
    print(f"ACT:   {entry['act'][:150]}")
    observe_preview = entry["observe"][:200]
    print(f"OBSERVE: {observe_preview}...")
    print()

In [ ]:
# Final response
print("=== REACT AGENT FINAL RESPONSE ===")
print(result["final_response"])

## You should now have

- [x] Seen the ReAct agent iterate through multiple tool calls
- [x] All five tools exercised in a single query
- [x] Every finding cites a specific FDA label section by set_id
- [x] Severity-first output (CRITICAL findings first)
- [x] Understood the difference between ungrounded and grounded medical AI

**Next:** `99_teardown.ipynb`